### Name: Adarsh Yadav

### Roll No:23107001
### Class:AIDS-A

### Title: Implement a basic AI prototype using TensorFlow/PyTorch for an application and Train the model with relevant datasets and evaluate its performance.

---

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Sem 6/AIPD/Dataset/news.csv")

print(df.head())
print("Total records:", len(df))

          idd                                              title  \
0  Fq+C96tcx+  ‘A target on Roe v. Wade ’: Oklahoma bill maki...   
1  bHUqK!pgmv  Study: women had to drive 4 times farther afte...   
2  4Y4Ubf%aTi        Trump, Clinton clash in dueling DC speeches   
3  _CoY89SJ@K  Grand jury in Texas indicts activists behind P...   
4  +rJHoRQVLe  As Reproductive Rights Hang In The Balance, De...   

                                                text label  
0  UPDATE: Gov. Fallin vetoed the bill on Friday....  REAL  
1  Ever since Texas laws closed about half of the...  REAL  
2  Donald Trump and Hillary Clinton, now at the s...  REAL  
3  A Houston grand jury investigating criminal al...  REAL  
4  WASHINGTON -- Forty-three years after the Supr...  REAL  
Total records: 4594


In [ ]:
df.head()

,idd,title,text,label
0,Fq+C96tcx+,‘A target on Roe v. Wade ’: Oklahoma bill maki...,UPDATE: Gov. Fallin vetoed the bill on Friday....,REAL
1,bHUqK!pgmv,Study: women had to drive 4 times farther afte...,Ever since Texas laws closed about half of the...,REAL
2,4Y4Ubf%aTi,"Trump, Clinton clash in dueling DC speeches","Donald Trump and Hillary Clinton, now at the s...",REAL
3,_CoY89SJ@K,Grand jury in Texas indicts activists behind P...,A Houston grand jury investigating criminal al...,REAL
4,+rJHoRQVLe,"As Reproductive Rights Hang In The Balance, De...",WASHINGTON -- Forty-three years after the Supr...,REAL


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4594 entries, 0 to 4593
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   idd     4594 non-null   object
 1   title   4593 non-null   object
 2   text    4594 non-null   object
 3   label   4594 non-null   object
dtypes: object(4)
memory usage: 143.7+ KB


In [ ]:
# Convert labels to numeric: REAL=1, FAKE=0
df['label_num'] = df['label'].apply(lambda x: 1 if x.upper() == "REAL" else 0)

# Handle missing titles by filling with an empty string before combining
df['title'] = df['title'].fillna('')

# Combine title + text
df['content'] = df['title'] + " " + df['text']

X = df['content']
y = df['label_num']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
vectorizer = TfidfVectorizer(max_features=3000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

In [ ]:
df.head()

,idd,title,text,label,label_num,content
0,Fq+C96tcx+,‘A target on Roe v. Wade ’: Oklahoma bill maki...,UPDATE: Gov. Fallin vetoed the bill on Friday....,REAL,1,‘A target on Roe v. Wade ’: Oklahoma bill maki...
1,bHUqK!pgmv,Study: women had to drive 4 times farther afte...,Ever since Texas laws closed about half of the...,REAL,1,Study: women had to drive 4 times farther afte...
2,4Y4Ubf%aTi,"Trump, Clinton clash in dueling DC speeches","Donald Trump and Hillary Clinton, now at the s...",REAL,1,"Trump, Clinton clash in dueling DC speeches Do..."
3,_CoY89SJ@K,Grand jury in Texas indicts activists behind P...,A Houston grand jury investigating criminal al...,REAL,1,Grand jury in Texas indicts activists behind P...
4,+rJHoRQVLe,"As Reproductive Rights Hang In The Balance, De...",WASHINGTON -- Forty-three years after the Supr...,REAL,1,"As Reproductive Rights Hang In The Balance, De..."


In [ ]:
def predict_fake_or_real(text):
    vec = vectorizer.transform([text]).toarray()
    tensor = torch.tensor(vec, dtype=torch.float32)
    with torch.no_grad():
        score = model(tensor).item()
    return "REAL" if score > 0.5 else "FAKE"

# Example
print(predict_fake_or_real("Trump announces new policy in Texas"))

REAL


In [ ]:
class NewsClassifier(nn.Module):
    def __init__(self, input_size):
        super(NewsClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

input_size = X_train_vec.shape[1]
model = NewsClassifier(input_size)

# Define Loss and Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(model)
print("Loss function: Binary Cross Entropy")
print("Optimizer: Adam")

# Training Loop
num_epochs = 10
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluation
with torch.no_grad():
    model.eval()
    y_pred_proba = model(X_test_tensor)
    y_pred = (y_pred_proba > 0.5).float()

    accuracy = accuracy_score(y_test_tensor, y_pred)
    report = classification_report(y_test_tensor, y_pred)

    print(f'\nAccuracy: {accuracy:.4f}')
    print('Classification Report:')
    print(report)

NewsClassifier(
  (fc1): Linear(in_features=3000, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Loss function: Binary Cross Entropy
Optimizer: Adam
Epoch [2/10], Loss: 0.6900
Epoch [4/10], Loss: 0.6810
Epoch [6/10], Loss: 0.6698
Epoch [8/10], Loss: 0.6576
Epoch [10/10], Loss: 0.6444

Accuracy: 0.8662
Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.83      0.86       450
         1.0       0.85      0.90      0.87       469

    accuracy                           0.87       919
   macro avg       0.87      0.87      0.87       919
weighted avg       0.87      0.87      0.87       919

